In [ ]:
import laspy
import numpy as np
import pathlib
import os

import dask.bag as db

# Pre-processing: Splitting large LAZ files

## Set input parameters

In [ ]:
# desired max file size (in bytes)
max_filesize = 250 * 2**20 # 250 MB

In [ ]:
# input path (original laz files)
input_path = pathlib.Path("/project/lidarac/Share/Workshop/ahn/laz")

# output path (split into smaller files)
output_path = pathlib.Path("/project/lidarac/Share/users/fnattino/split")

In [ ]:
laz_files = list(input_path.glob("*.LAZ"))
print(f"Splitting {len(laz_files)} LAZ files")

## Setup Cluster

Setup Dask cluster used for splitting.

In [ ]:
from dask.distributed import Client

client = Client("tcp://10.0.0.52:33961")
client

## Splitting

In [ ]:
# make sure output dir exists
output_path.mkdir(exist_ok=True)

In [ ]:
# approximate compression factor
LAZ_COMPRESSION_FACTOR = 7

def save_chunk_to_laz_file(in_filename,
                           out_filename,
                           offset,
                           n_points):
    """Read points from a LAS/LAZ file and write them to a new file."""
    points = np.array([])
    with laspy.open(in_filename) as in_file:
        with laspy.open(out_filename,
                        mode="w",
                        header=in_file.header) as out_file:
            in_file.seek(offset)
            points = in_file.read_points(n_points)
            out_file.write_points(points)
    return len(points)

def split_strategy(filepath, max_filesize, out_dir="./"):
    """Set up splitting strategy for a LAS/LAZ file."""
    with laspy.open(filepath) as f:
        bytes_per_point = (
            f.header.point_format.num_standard_bytes +
            f.header.point_format.num_extra_bytes
        )
        n_points = f.header.point_count
    n_points_target = int(
        max_filesize * LAZ_COMPRESSION_FACTOR / bytes_per_point
    )
    filename = os.path.basename(filepath)
    stem, ext = os.path.splitext(filename)
    return [
        (filepath, f"{out_dir}/{stem}-{n:02d}{ext}", offset, n_points_target)
        for n, offset in enumerate(range(0, n_points, n_points_target))
    ]

In [ ]:
# set up calculation
files = db.from_sequence(laz_files)
input_args = files \
    .map(
        split_strategy,
        max_filesize=max_filesize,
        out_dir=output_path.as_posix(),
    ) \
    .flatten() \
    .unzip(4)  # unpack input arguments
res = db.map(save_chunk_to_laz_file, *input_args)

In [ ]:
tot_points = res.compute()

In [ ]:
# splitted points
sum(tot_points)

## Terminate cluster

In [ ]:
# client.shutdown()